# ListingLens 🏠🔍
**An eval harness for AI listing-photo tagging.**

This notebook uses a vision LLM to tag interior room photos with structured listing data, then scores how trustworthy those tags are against a hand-built golden set of 30 images.

The point isn't the tagging — it's measuring *where* the model is wrong, and whether its confidence can be trusted.

## 1. Setup
Connect to the Gemini vision API. The API key is stored as a Colab **Secret** named `GEMINI_API_KEY` (left sidebar → 🔑), so it never appears in the code.

In [ ]:
# Install the Google GenAI SDK and connect
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata
import json, time, os
import pandas as pd

MODEL = "gemini-2.5-flash-lite"   # vision model; use gemini-2.5-flash for higher quality
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
print("Connected ✓")

## 2. Connect the images
The 30 photos and `golden_set.csv` live in a Google Drive folder. Mounting Drive lets the notebook read them directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

IMAGE_FOLDER = "/content/drive/MyDrive/Image_folder"   # <-- set to your folder

files = sorted(f for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith((".jpg", ".jpeg", ".png")))
print(f"Found {len(files)} images.")

## 3. The tagging function
Sends one image plus an instruction to the model and gets back structured JSON. The prompt tells the model to be honest and lower its confidence when unsure, rather than guessing. It retries automatically if the API is briefly busy or rate-limited.

In [ ]:
from google.genai import errors

PROMPT = """You are tagging an interior real-estate listing photo. Return ONLY valid JSON (no markdown, no extra text) with exactly these keys:
- "is_a_room_photo": true if this is a photo of an interior room; false if it is anything else (an exterior, a floorplan drawing, a collage, etc.)
- "room_type": one of [living room, bedroom, kitchen, bathroom, dining room, home office, balcony, hallway]. If it IS a room but you cannot tell the type, use "room". If it is NOT a room, use "none".
- "style_tags": list of 1-2 style words (e.g. modern, scandinavian). Use ["unclear"] if you cannot tell.
- "key_objects": list of clearly visible, notable furniture/fixtures. Do NOT list anything you are not sure is present. Use [] if none.
- "condition": one of [furnished, semi-furnished, unfurnished, under_renovation, staged].
- "description": one short sentence.
- "confidence": your own certainty, integer 0-100.
Be honest: if the image is dark, blurry, empty, or not a room, say so and LOWER your confidence rather than guessing."""

def tag_image(path, retries=6):
    mime = "image/png" if path.lower().endswith(".png") else "image/jpeg"
    with open(path, "rb") as f:
        image_bytes = f.read()
    for attempt in range(retries):
        try:
            start = time.time()
            response = client.models.generate_content(
                model=MODEL,
                contents=[types.Part.from_bytes(data=image_bytes, mime_type=mime), PROMPT],
                config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0),
            )
            data = json.loads(response.text)
            data["_latency_sec"] = round(time.time() - start, 2)
            return data
        except (errors.ServerError, errors.ClientError) as e:
            if any(x in str(e) for x in ["503", "429", "UNAVAILABLE", "RESOURCE_EXHAUSTED"]):
                wait = min(60, 10 * (attempt + 1))
                print(f"  busy (try {attempt+1}/{retries}) — waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("retries exhausted")

## 4. Tag every image
Runs the tagger across all 30 photos and saves the results to `ai_results.csv`. It saves after each image and skips ones already done, so a single failure never loses progress.

In [ ]:
out_path = f"{IMAGE_FOLDER}/ai_results.csv"

if os.path.exists(out_path):
    prev = pd.read_csv(out_path)
    good = prev[prev["ai_room_type"].notna()]
    rows = good.to_dict("records")
    done = set(good["filename"])
    print(f"Resuming — {len(done)} already done.")
else:
    rows, done = [], set()

for fname in files:
    if fname in done:
        continue
    print(f"Tagging {fname} ...")
    try:
        data = tag_image(f"{IMAGE_FOLDER}/{fname}")
    except Exception as e:
        print(f"  ⚠️ {fname} failed: {e}")
        data = {}
    rows.append({
        "filename": fname,
        "ai_is_a_room_photo": data.get("is_a_room_photo"),
        "ai_room_type": data.get("room_type"),
        "ai_style": "; ".join(data.get("style_tags", [])),
        "ai_objects": "; ".join(data.get("key_objects", [])),
        "ai_condition": data.get("condition"),
        "ai_description": data.get("description"),
        "ai_confidence": data.get("confidence"),
        "latency_sec": data.get("_latency_sec"),
    })
    pd.DataFrame(rows).to_csv(out_path, index=False)
    time.sleep(5)

ai_df = pd.DataFrame(rows)
print(f"\nDone! {len(ai_df)} rows saved.")
ai_df

## 5. The eval
Compares the model's output to the hand-built golden set and reports the scorecard: is-it-a-room accuracy, room-type accuracy, condition accuracy, object precision/recall, confidence calibration, latency, and the confidently-wrong cases. Object matching is synonym/substring tolerant so "sofa" and "sectional sofa" count as a hit.

In [ ]:
import re

golden = pd.read_csv(f"{IMAGE_FOLDER}/golden_set.csv", encoding="latin-1")
ai     = pd.read_csv(f"{IMAGE_FOLDER}/ai_results.csv")
df = golden.merge(ai, on="filename", how="left")

def norm(s): return re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", str(s).lower())).strip()
def singular(w): return w[:-1] if w.endswith("s") and len(w) > 3 else w
def head(o):
    o = norm(o); return singular(o.split()[-1]) if o else ""
def to_set(s):
    return [] if pd.isna(s) else [x.strip() for x in str(s).split(";") if x.strip() and norm(x) != "none"]
def obj_match(g, a):
    gn, an = norm(g), norm(a)
    if not gn or not an: return False
    if gn == an or gn in an or an in gn: return True
    return head(g) == head(a) and head(g) != ""

s = df[df["ai_room_type"].notna()].copy()
s["ai_room"]   = s["ai_is_a_room_photo"].astype(str).str.lower().eq("true")
s["gold_room"] = s["golden_room_type"].apply(lambda x: norm(x) != "none")

def type_ok(r):
    g, a = norm(r["golden_room_type"]), norm(r["ai_room_type"])
    if g == "none": return a == "none" or not r["ai_room"]
    if g == "room": return r["ai_room"]
    return g == a
s["room_type_ok"] = s.apply(type_ok, axis=1)
s["cond_ok"] = s.apply(lambda r: norm(r["golden_condition"]) == norm(r["ai_condition"]), axis=1)

P, R = [], []
for _, r in s.iterrows():
    g, a = to_set(r["golden_objects"]), to_set(r["ai_objects"])
    if not g and not a: continue
    R.append(sum(any(obj_match(x, y) for y in a) for x in g) / len(g) if g else 1.0)
    P.append(sum(any(obj_match(x, y) for x in g) for y in a) / len(a) if a else 1.0)
prec, rec = sum(P)/len(P), sum(R)/len(R)

spec = s[~s["golden_room_type"].apply(lambda x: norm(x) in ("room", "none"))]
s["conf_bucket"] = s["ai_confidence"].apply(lambda c: "85-100" if c >= 85 else ("60-84" if c >= 60 else "0-59"))
s["overconfident_wrong"] = (s["ai_confidence"] >= 85) & (~s["room_type_ok"])

print(f"Scored: {len(s)}/30")
print(f"Is-a-room accuracy        : {(s['gold_room']==s['ai_room']).mean()*100:.0f}%")
print(f"Room-type accuracy (spec) : {spec['room_type_ok'].mean()*100:.0f}%")
print(f"Condition accuracy        : {s['cond_ok'].mean()*100:.0f}%")
print(f"Object precision/recall/F1: {prec*100:.0f}% / {rec*100:.0f}% / {2*prec*rec/(prec+rec)*100:.0f}%")
print(f"Avg latency               : {s['latency_sec'].mean():.2f}s\n")
print("Confidence calibration:")
print(s.groupby("conf_bucket")["room_type_ok"].agg(["size", "mean"]).to_string())
print(f"\nConfidently-wrong (conf>=85 but incorrect): {int(s['overconfident_wrong'].sum())}")
for _, r in s[s["overconfident_wrong"]].iterrows():
    print(f"  {r['filename']}: golden={r['golden_room_type']} | AI={r['ai_room_type']} @ conf {r['ai_confidence']:.0f}")

s.to_csv(f"{IMAGE_FOLDER}/scored.csv", index=False)
print("\nSaved scored.csv")

## Key finding
The model lowered its confidence correctly on genuinely hard images (the blurry photo → 10, the mirror room → 30) but was *confidently* wrong on two ordinary-looking photos. So the dangerous errors hide in confident, normal outputs — exactly where a human reviewer wouldn't think to look. A confidence threshold alone wouldn't catch them, which is why the review design also spot-checks high-confidence outputs.